In [17]:
%pip install python-dotenv langchain langchain-classic langchain_core langchain-tavily langchain-community langchain-openai openai langchain-google-genai langchain-anthropic langchain-store

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement langchain-store (from versions: none)
ERROR: No matching distribution found for langchain-store


In [57]:
from typing import Any
from langchain_core.messages import BaseMessage
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore
from langchain_anthropic import ChatAnthropic



In [58]:
from dotenv import load_dotenv
import os

load_dotenv()

anthropic_key = os.getenv("ANTHROPIC_API_KEY")
print(anthropic_key[:5])

sk-an


In [69]:

@tool
def get_user_info(userId: str, runtime: ToolRuntime) -> str:
    """Lookup user information."""
    store = runtime.store
    user_info = store.get("users", userId)
    return str(user_info.value) if user_info else "User not found"

@tool
def save_user_info(userId: str, userInf: dict[str, Any], runtime: ToolRuntime) -> str:
    """Save user information."""
    store = runtime.store
    store.put("users", userId, userInf)
    return f"User {userId} Successfully saved"


In [65]:
store = InMemoryStore()


In [66]:
anthropic_model = ChatAnthropic(
    model="claude-sonnet-4-5",
    temperature=0
)

In [67]:
agent = create_agent(
    anthropic_model,
    tools=[get_user_info, save_user_info],
    store=store
)


In [68]:

# First session: save user info
agent.invoke({
    "messages": [{"role": "user", "content": "Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev"}]
})

# Second session: get user info
agent.invoke({
    "messages": [{"role": "user", "content": "Get user info for user with id 'abc123'"}]
})

AttributeError: 'InMemoryStore' object has no attribute 'set'